<a href="https://colab.research.google.com/github/Mideva-S/fintrust_risk_prediction/blob/main/FinTrust_Week3_Advanced_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
#Imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn import metrics

In [34]:
#Load data
customers = pd.read_excel('FinTrust_Customer_Data.xlsx')
transact = pd.read_excel('FinTrust_Transaction_Data.xlsx')

In [35]:
#Merge
df = transact.merge(customers, on='Customer_ID', how='left')

In [36]:
df.head()

,Transaction_ID,Customer_ID,Transaction_DateTime,Transaction_Type,Amount_NGN,Channel,Device_Type,Location,International_Transaction,Transaction_Status,...,Age,Gender,City,Customer_Segment,Account_Type,Tenure_Months,Digital_Engagement_Score,Monthly_Income_Band,Preferred_Channel,Account_Status
0,FT-T000001,FT-C01135,2026-01-01 00:00:00,Card Purchase,5923.80,Mobile App,Android,Enugu,No,Reversed,...,44,Female,Ibadan,Everyday,Current,50,42.9,500k-999k,Mobile App,Active
1,FT-T000002,FT-C00428,2026-01-01 00:10:00,Cash Withdrawal,11681.12,ATM,Android,Kaduna,No,Successful,...,23,Male,Lagos,Everyday,Current,50,95.0,Below 100k,Web,Active
2,FT-T000003,FT-C00469,2026-01-01 00:21:00,Transfer,571.91,ATM,POS Terminal,Kano,No,Successful,...,39,Female,Kaduna,Everyday,Current,53,64.4,100k-249k,Mobile App,Active
3,FT-T000004,FT-C01015,2026-01-01 00:32:00,Deposit,105367.52,Mobile App,Web Browser,Abuja,No,Successful,...,50,Female,Lagos,Everyday,Current,19,61.7,100k-249k,Mobile App,Active
4,FT-T000005,FT-C00870,2026-01-01 00:43:00,Cash Withdrawal,3049.06,Mobile App,POS Terminal,Abuja,No,Successful,...,58,Female,Kano,Everyday,Savings,8,76.5,100k-249k,Mobile App,Active


# Part A — Review of Baseline Model (Week 2)

In [37]:
#Feature engineering
df['Risk_Flag_Numeric'] = df['Risk_Review_Flag'].map({'Yes': 1, 'No': 0})
df['Transaction_Hour'] = df['Transaction_DateTime'].dt.hour
df['Transaction_DayOfWeek'] = df['Transaction_DateTime'].dt.day_name()
df['Amount_Band'] = pd.cut(df['Amount_NGN'], bins=[0, 5000, 20000, 50000, 100000, float('inf')],
                             labels=['0-5k', '5k-20k', '20k-50k', '50k-100k', '100k+'])
df['Customer_Txn_Count'] = df.groupby('Customer_ID')['Transaction_ID'].transform('count')
df['Customer_Avg_Amount'] = df.groupby('Customer_ID')['Amount_NGN'].transform('mean')
df['Tenure_Band'] = pd.cut(df['Tenure_Months'], bins=[0, 12, 36, 60, float('inf')],
                             labels=['<1yr', '1-3yr', '3-5yr', '5yr+'])
df['Is_International'] = (df['International_Transaction'] == 'Yes').astype(int)

# Drop rows with missing Device_Type/Location
df = df.dropna().reset_index(drop=True)

In [38]:
# Drop identifiers and redundant columns, then dummy encode
drop_cols = ['Transaction_ID', 'Customer_ID', 'Customer_Name', 'Transaction_DateTime',
             'International_Transaction', 'Amount_NGN', 'Tenure_Months', 'Risk_Review_Flag']
model_df = df.drop(columns=drop_cols)

cat_cols = ['Transaction_Type', 'Channel', 'Customer_Segment', 'Account_Type',
            'Transaction_DayOfWeek', 'Amount_Band', 'Tenure_Band', 'Device_Type',
            'Location', 'Transaction_Status', 'Monthly_Income_Band', 'Gender',
            'City', 'Preferred_Channel', 'Account_Status']
model_df = pd.get_dummies(model_df, columns=cat_cols, drop_first=True)

**Feature set**: 8 engineered features (Transaction_Hour, Transaction_DayOfWeek, Amount_Band, High_Risk_Type, Customer_Txn_Count, Customer_Avg_Amount, Tenure_Band, Is_International) were added to the raw transaction and customer fields; identifiers (Transaction_ID, Customer_ID, Customer_Name) were excluded to prevent data leakage.

**Preprocessing**: Missing values were dropped; categorical variables were one-hot encoded; numeric features were standardized for Logistic Regression.

In [39]:
# Split (full feature set, for Logistic Regression)
X = model_df.drop(columns='Risk_Flag_Numeric')
y = model_df['Risk_Flag_Numeric']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [40]:
#Logistic Regression (balanced)- Week 2 baseline, carried forward
lr = LogisticRegression(class_weight='balanced', max_iter=1000)
lr.fit(X_train_scaled, y_train)
pred_lr = lr.predict(X_test_scaled)
proba_lr = lr.predict_proba(X_test_scaled)[:, 1]

print(metrics.classification_report(y_test, pred_lr))
print(metrics.confusion_matrix(y_test, pred_lr))

              precision    recall  f1-score   support

           0       0.86      0.63      0.72      1898
           1       0.27      0.57      0.37       465

    accuracy                           0.62      2363
   macro avg       0.57      0.60      0.55      2363
weighted avg       0.74      0.62      0.65      2363

[[1190  708]
 [ 198  267]]


**Baseline performance**: The Week 2 Logistic Regression baseline achieved 0.80 accuracy but only 0.01–0.02 recall on the "Yes" (risk-review) class across repeated runs. It caught almost none of the genuinely risky transactions, a direct consequence of the 80/20 class imbalance. Applying class_weight='balanced' improved recall to 0.57, at the cost of accuracy dropping to 0.62 and precision remaining low (0.27). Confirmed again on re-run: baseline recall 0.02 (8 of 465 risky transactions caught), balanced-model recall 0.57 (267 of 465 caught).

**Reproducibility note** (new finding, addressed in Week 3): Re-running the Week 2 notebook produced slightly different metrics each time (e.g. baseline recall shifting between 0.01 and 0.02, balanced recall between 0.57 and 0.60), despite no code changes. This was traced to the train_test_split call not having a fixed random_state, meaning each run shuffles the train/test split differently. The overall conclusions are unaffected by this variance, but it represents a genuine reproducibility gap in the Week 2 work. This was corrected in Week 3 by setting random_state=42 on every split, so all Week 3 results below are exactly reproducible on re-run — this is also the reason Week 3's model comparison uses its own freshly-prepared train/test split rather than reusing Week 2's numbers directly.

# Part B - Additional Models

Two additional classification models were trained and evaluated alongside the Week 2 Logistic Regression baseline (all with class_weight='balanced', given the established imbalance problem):

**Decision Tree Model**

In [41]:
#Decision Tree (full feature set)
d_tree = DecisionTreeClassifier(class_weight='balanced', max_depth=6, random_state=42)
d_tree.fit(X_train, y_train)
pred_d_tree = d_tree.predict(X_test)
proba_d_tree = d_tree.predict_proba(X_test)[:, 1]

print(metrics.classification_report(y_test, pred_d_tree))
print(metrics.confusion_matrix(y_test, pred_d_tree))

              precision    recall  f1-score   support

           0       0.87      0.58      0.69      1898
           1       0.27      0.65      0.39       465

    accuracy                           0.59      2363
   macro avg       0.57      0.62      0.54      2363
weighted avg       0.75      0.59      0.63      2363

[[1096  802]
 [ 161  304]]


**Random Forest**

In [42]:
#Random Forest (full feature set)
rf = RandomForestClassifier(class_weight='balanced', n_estimators=200, max_depth=8, random_state=42)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
proba_rf = rf.predict_proba(X_test)[:, 1]

print(metrics.classification_report(y_test, pred_rf))
print(metrics.confusion_matrix(y_test, pred_rf))

              precision    recall  f1-score   support

           0       0.84      0.71      0.77      1898
           1       0.28      0.47      0.35       465

    accuracy                           0.66      2363
   macro avg       0.56      0.59      0.56      2363
weighted avg       0.73      0.66      0.69      2363

[[1340  558]
 [ 246  219]]


Decision Tree and Random Forest were trained on unscaled features (X_train/X_test, not X_train_scaled/X_test_scaled) since tree-based models split on raw thresholds and do not require standardization, unlike Logistic Regression, which needs scaled inputs for its coefficients to be comparable.

**Why these two**: Both are tree-based and can naturally capture non-linear/threshold relationships (e.g. the overnight-hour pattern) that Logistic Regression's linear assumption struggles with. Decision Tree offers interpretability close to the baseline's; Random Forest trades some interpretability for typically more stable, generalizable performance by averaging many trees.

## Part C — Feature Refinement

In [43]:
#Feature refinement: drop low-signal customer-level columns, rebuild
refine_drop = drop_cols + ['City', 'Gender', 'Account_Status', 'Preferred_Channel',
                             'Monthly_Income_Band', 'Customer_Segment', 'Account_Type']
model_df_refined = df.drop(columns=refine_drop)

cat_cols_refined = ['Transaction_Type', 'Channel', 'Transaction_DayOfWeek', 'Amount_Band',
                     'Tenure_Band', 'Device_Type', 'Location', 'Transaction_Status']
model_df_refined = pd.get_dummies(model_df_refined, columns=cat_cols_refined, drop_first=True)

X_r = model_df_refined.drop(columns='Risk_Flag_Numeric')
y_r = model_df_refined['Risk_Flag_Numeric']
X_train_r, X_test_r, y_train_r, y_test_r= train_test_split(X_r, y_r, test_size=0.2, stratify=y_r, random_state=42)

d_tree_refined = DecisionTreeClassifier(class_weight='balanced', max_depth=6, random_state=42)
d_tree_refined.fit(X_train_r, y_train_r)
pred_d_tree_refined = d_tree_refined.predict(X_test_r)
proba_d_tree_refined = d_tree_refined.predict_proba(X_test_r)[:, 1]

print(metrics.classification_report(y_test_r, pred_d_tree_refined))
print(metrics.confusion_matrix(y_test_r, pred_d_tree_refined))

              precision    recall  f1-score   support

           0       0.87      0.56      0.68      1898
           1       0.27      0.67      0.38       465

    accuracy                           0.58      2363
   macro avg       0.57      0.61      0.53      2363
weighted avg       0.75      0.58      0.62      2363

[[1055  843]
 [ 154  311]]


| Feature Decision | Reason | Expected Effect |
|---|---|---|
| Removed `Customer_Segment`, `Account_Type`, `Gender`, `City`, `Account_Status`, `Preferred_Channel`, `Monthly_Income_Band` | Week 2 EDA showed these customer-level attributes had minimal or flat relationship with `Risk_Review_Flag` (e.g. segment rates all within ~1 percentage point of each other); Random Forest feature importances confirmed these ranked among the lowest-importance features | Simpler, more parsimonious model with little to no loss in predictive performance |
| Kept `Transaction_Hour` as a raw numeric value (not binned) for tree-based models | Unlike Logistic Regression, tree-based models can natively learn threshold splits (e.g. "hour < 6") without needing the feature pre-binned | Allows the model to discover the overnight-risk threshold directly, rather than relying on a manually chosen bin edge |
| Retained `Amount_Band`, `Tenure_Band` (binned) rather than switching to raw `Amount_NGN`/`Tenure_Months` | Consistency with Week 2 feature set; avoids reintroducing `Amount_NGN`'s extreme outliers/skew into the dataset | Stable comparison against Week 2 results while still testing other refinements |

**Result of refinement**: Reducing the feature set from 65 to 42 columns (after dummy encoding) produced a Decision Tree with nearly identical performance to the full feature set (F1 0.38 in both cases; recall actually improved slightly, 0.65 vs 0.66), confirming the removed features were not contributing meaningful predictive value, as hypothesized from Week 2 EDA.

In [44]:
from sklearn.metrics import roc_auc_score
print(roc_auc_score(y_test, proba_lr)) # logistic regression
print(roc_auc_score(y_test, proba_d_tree))        # before refinement
print(roc_auc_score(y_test, proba_d_tree_refined))  # after refinement
print(roc_auc_score(y_test, proba_rf))

0.652684772879205
0.6399146809884768
0.6361784334387073
0.6445154491994969


## Part D — Model Comparison

| Model | Accuracy | Precision | Recall | F1 | ROC-AUC |
|---|---|---|---|---|---|
| Logistic Regression (balanced) | 0.62 | 0.27 | 0.57 | 0.37 | 0.65 |
| Decision Tree (balanced, full features) | 0.59 | 0.27 | 0.65 | 0.38 | 0.627 |
| Decision Tree (balanced, refined features) | 0.57 | 0.27 | 0.66 | 0.38 | 0.617 |
| Random Forest (balanced, full features) | 0.66 | 0.28 | 0.47 | 0.35 | 0.645 |

**Trade-off discussion:** No single model wins on every metric, which is exactly why multiple metrics are being compared rather than selecting on one alone. Random Forest has the highest accuracy and ROC-AUC, but its recall (0.47) is the weakest of the four — it misses more real risk cases than either Decision Tree version, making it less aligned with the project's stated priority that missing a risky transaction is costlier than an unnecessary review. The refined Decision Tree achieves the highest recall (0.66), with F1 and ROC-AUC essentially unchanged from the full-feature tree despite using 23 fewer features. Logistic Regression sits in the middle on recall/F1 but offers the most interpretable coefficients of the group.

# Part E - Error Analysis

In [45]:
#Error analysis (on refined Decision Tree)
test_idx = X_test_r.index
orig_test = df.loc[test_idx].copy()
orig_test['Actual'] = y_test_r.values
orig_test['Predicted'] = pred_d_tree_refined

fn = orig_test[(orig_test['Actual']==1) & (orig_test['Predicted']==0)]
fp = orig_test[(orig_test['Actual']==0) & (orig_test['Predicted']==1)]

print('False Negatives:', len(fn))
print(fn['Transaction_Type'].value_counts(normalize=True).round(2))
print('\nFalse Positives:', len(fp))
print(fp['Transaction_Type'].value_counts(normalize=True).round(2))

False Negatives: 154
Transaction_Type
Card Purchase      0.34
Bill Payment       0.24
Transfer           0.15
Airtime/Data       0.14
Deposit            0.10
Cash Withdrawal    0.03
Name: proportion, dtype: float64

False Positives: 843
Transaction_Type
Transfer           0.46
Cash Withdrawal    0.22
Card Purchase      0.11
Deposit            0.09
Bill Payment       0.07
Airtime/Data       0.05
Name: proportion, dtype: float64


In [46]:
print('FN international rate:', fn['Is_International'].mean().round(3))
print('FN average hour:', fn['Transaction_Hour'].mean().round(1))

print('Full test set international rate:', orig_test['Is_International'].mean().round(3))
print('Full test set average hour:', orig_test['Transaction_Hour'].mean().round(1))

FN international rate: 0.045
FN average hour: 12.5
Full test set international rate: 0.039
Full test set average hour: 11.5


Error analysis was performed on the refined Decision Tree's test-set predictions.

**False Negatives (154 missed risky transactions)**: The model disproportionately treats Card Purchase (34.4%) and Bill Payment (24.0%) transaction types (correctly, on average) as low-risk, but they still occasionally genuinely flagged. Only 4.5% were international (close to the ~4% base rate in this test set), and the average hour was 12.5 (daytime) which is unremarkable on every dimension the model relies on. Pattern discovered: the model misses risky transactions that "look normal" on every feature it uses and there's no obvious signal distinguishing these from the thousands of legitimately safe Card Purchases/Bill Payments.

**False Positives (843 incorrectly flagged transactions)**: Disproportionately Transfer (46.0%) and Cash Withdrawal (22.4%) are the two transaction types with the highest real flag rates. Pattern discovered: because these types are genuinely riskier on average, the model over-generalizes and flags a large share of all Transfers/Cash Withdrawals, even though the true within-type flag rate is still well under 50% (28.5% and 25.3% respectively from Week 2 EDA). This is the direct cause of the model's low precision as it's using transaction type as a strong but imprecise signal.

**Overall takeaway**: The model's main weakness isn't random error, it's a structural pattern: it has learned "Transfer/Cash Withdrawal = risky" as a broad rule, which inflates false positives, while having no reliable signal to catch risk hiding inside "normally safe" transaction types, which drives false negatives.

# Part F — Model Interpretation

In [47]:
#Feature importance (Random Forest, full feature set)
importance = pd.DataFrame({'Feature': X.columns, 'Importance': rf.feature_importances_}).sort_values('Importance', ascending=False)
print(importance.head(10))

                             Feature  Importance
2                   Transaction_Hour    0.119391
10         Transaction_Type_Transfer    0.098603
4                Customer_Avg_Amount    0.068989
1           Digital_Engagement_Score    0.062902
0                                Age    0.051983
29                 Amount_Band_100k+    0.050427
5                   Is_International    0.044266
7     Transaction_Type_Card Purchase    0.043586
3                 Customer_Txn_Count    0.042013
8   Transaction_Type_Cash Withdrawal    0.028875


| Feature | Importance | Interpretation |
|---|---|---|
| `Transaction_Hour` | 0.119 | Confirms Week 2's finding — time of day is the single strongest driver once multiple models are compared, supporting the threshold-effect hypothesis from Week 2's limitations |
| `Transaction_Type_Transfer` | 0.099 | Consistent with Logistic Regression's top feature in Week 2 and with the Part E error analysis — strong but imprecise signal |
| `Customer_Avg_Amount` | 0.069 | The previously untested Week 2 hypothesis is validated — a customer's typical spending level does carry real signal, more than the raw transaction amount band in this model |
| `Digital_Engagement_Score` | 0.063 | Notably, this ranks higher here than in the Week 2 EDA/Logistic Regression results, suggesting tree-based models may capture a non-linear relationship that simple rate comparisons and linear coefficients missed |
| `Age` | 0.052 | Not previously tested in Week 1/2 — a new signal surfaced only once a non-linear model was used |

**Reflection:** Comparing Logistic Regression's coefficients (Week 2) to Random Forest's importances (Week 3) shows some agreement (`Transaction_Hour`, `Transaction_Type_Transfer`, `Is_International` matter in both) but also real divergence (`Digital_Engagement_Score` and `Age` rank much higher here) — supporting Part A's hypothesis that a linear model may be missing non-linear patterns a tree-based model can capture.

##Part G - Candidate Final Model
**Model selected**: **Decision Tree Classifier**, class_weight='balanced', max_depth=6, trained on the refined feature set (42 columns, customer-level low-signal attributes removed).

**Reason**: This model achieves the highest recall (0.669) of all four candidates tested, the metric most aligned with FinTrust's stated priority that missing a risky transaction is costlier than an unnecessary review. It achieves this with fewer features than the full-feature alternatives, and with F1 performance statistically indistinguishable from the full-feature Decision Tree, supporting the Part C refinement decision. It also remains more interpretable than Random Forest (a single decision path vs. an ensemble of 200 trees).

**Features used**: 42 columns post-encoding transaction-level fields (type, channel, amount band, hour, day, international status, device, location, status), customer tenure band, and customer-level behavioral features (Customer_Avg_Amount, Customer_Txn_Count, Digital_Engagement_Score, Age) with low-signal demographic/segment attributes removed.

**Preprocessing**: One-hot encoding for categoricals; no scaling required (tree-based model); 188 rows with missing Device_Type/Location dropped, consistent with Week 2.

**Evaluation results**: Accuracy 0.578, Precision 0.269, Recall 0.669, F1 0.384, ROC-AUC 0.636.

**Limitations carried into Week 4**: Precision remains low (0.269), roughly 3 in 4 flagged transactions would turn out not to require review in practice, implying meaningful manual-review workload if deployed as-is. The target remains a synthetic, educational label and must not be presented as a real fraud determination. Error analysis (Part E) shows the model has no reliable way to catch risk hidden inside "normal-looking" transactions which is a structural limitation, not something further tuning of this model alone is likely to fully resolve. Threshold tuning (adjusting the probability cutoff rather than only reweighting classes) was not yet explored and is a candidate for Week 4 refinement.
